In [ ]:
# -----------------------------导入python库------------------------------------
import os
import netCDF4 as nc
import pandas as pd
import numpy as np
import numpy.ma as ma
import xarray as xr
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.style as mplstyle
import cartopy.crs as ccrs
import netCDF4 as nc
from netCDF4 import Dataset
import shapefile
import time
from datetime import datetime, timedelta

import cmaps
BASE_PATH = "/home/yfdong/data/work/SMP"
DATA_PATH = "/raid61/yfdong/data/work/SMP"
import sys
# 将目标模块所在的路径添加到 sys.path 中
sys.path.append(os.path.join(BASE_PATH, "code", "Library"))
from MeteoVarPlot import draw_2DVAR, draw_Bias, draw_VARtemporal, draw_Bias_subplot,draw_2DVAR_subplot ,draw_2DuvVAR ,draw_uvBias_subplot ,draw_2DuvVAR_subplot
from MeteoChartPlot import add_TickGrid
from PreprocessVar import (
    get_MONTH_abbr, get_pentad, get_letter,
    preprocess_var, read_var,Events_configs,
    Read_VarsFromVarsDict, experiments_types,return_Event_Date,
    load_geodata, SubsetDomainConfigure ,create_domain_mask,get_letter,
    
)
Era5Land_VarResults={}
Wrfout_VarResults={}
Obs_DataPath = os.path.join(DATA_PATH, "Evaluation/Obs")
Sim_DataPath = os.path.join(DATA_PATH, "Evaluation/WRF")

# -----------------------------读取数据------------------------------------
LSM_VarList = ['T2m', 'PSFC', 'ET', 'Q2m', 'Wind10m', 'Prec'] #'Wind10m',
# LSM_VarList = ['SH']
# 读取地形数据
# LAT, LON, LandMask, DEM = load_geodata(GeoPath =os.path.join(DATA_PATH, "Domain/geo_em.d01.nc") ,IF_LandMask=True)
ds = xr.open_dataset('/raid61/yfdong/data/work/SMP/Evaluation/WRF/FSM_ET_10km.2024.nc')
lat, lon = ds['lat'], ds['lon']
lon_grid, lat_grid = np.meshgrid(lon, lat)
et_2d = ds['ET'][0, :, :].values
LandMask = xr.where(et_2d != 0, 1.0, np.nan)
Timedelta = 24  # 选取的wrfout时间间隔（hour）
CalibrationStatus = 'def'
LAYER = 0
START_YEAR = 2000
END_YEAR = 2024
YearNum = END_YEAR - START_YEAR + 1
StartReadMonth, EndReadMonth = 5, 8
StartDay, EndDay = 6, 31
CtrlName, ExpName = 'Ctrl', 'FSM'

In [ ]:
obs_var_mapping = {
    'ET': 'E',
    # 'SM': 'swvl1',  # 土壤湿度第一层
    'SH': 'H',
    'Q2m': 'shum',
    'T2m': 'temp',
    'PSFC': 'pres',
    'Wind10m': 'wind',
    'Prec': 'cmorph',
}
sim_var_mapping = {
    # 'SM': 'SH2O_Layer1',  # 土壤湿度第一层
    'ET': 'ET',
    'SH': 'SH',
    'Q2m': 'Q2',
    'T2m': 'T2',
    'PSFC': 'PSFC',
    'Wind10m': 'Wind10',
    'Prec': 'RAINNC',
}
obs_folder_mapping = {
    'ET': 'GLEAM',
    'SH': 'GLEAM', #GLEAM_H_wrfout_grid.2021
    'Q2m': 'CMFD', 
    'T2m': 'CMFD',
    'PSFC': 'CMFD', #CMFD_pres_wrfout_grid.2018.nc
    'Wind10m': 'CMFD',
    'Prec': 'CMORPH', #CMORPH_cmorph_wrfout_grid.2000.nc
}
sim_folder_mapping = {
    'ET': 'ET',
    'LH': 'LH',
    'SH': 'SH',
    'SSM': 'SH2O_Layer1',  # 土壤湿度第一层
    'RSM': 'SH2O_Layer3',  # 土壤湿度第三层
    # 'SST': 'TSLB_Layer1',  # 土壤温度第一层
    'Q2m': 'Q2',
    'T2m': 'T2',
    'PSFC': 'PSFC',
    'Wind10m': 'Wind10',
    'Prec':'RAINNC',
}

obs_var_ScaleFactor = {
    'ET': 1,
    'SH': 1,
    'Q2m': 1000, # kg/kg -> g/kg
    'Wind10m': 1,
    'T2m': 1,
    'Prec': 1, #
    'PSFC': 0.01, # Pa -> hPa
}
sim_var_ScaleFactor = {
    'ET': 1,
    'SH': 1,
    'Q2m': 1, # 
    'Wind10m': 1,
    'T2m': 1,
    'Prec': 24, # mm/h -> mm/d
    'PSFC': 1, 
}

VarUnit = {
    'SM': 'm³/m³',
    'ET': 'mm/day',
    'SH': 'W/m²',
    'Q2m': 'g/kg',
    'T2m': 'K',
    'PSFC': 'hPa',
    'Wind10m': 'm/s',
    'Prec': 'mm/day',
}

SubsetRegions = {
    'HHP':[35, 40, 114., 119],
    'LPT': [35, 40, 108, 114.],
    'QDW': [28.5, 35, 108, 112.5], # Qinling-Daba-Wushan
    'YHP': [28.5, 35, 112.5, 121], 
    'SEH': [23, 28.5, 110, 120] # Southeastern Hills
}
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error
# 根据序列Obs和Sim返回RMSE, BIAS, CORR和 KGE
def compute_metric(obs, sim, metric_name):
    # 质控（也可在外部做）
    obs = np.asarray(obs, dtype=np.float64)
    sim = np.asarray(sim, dtype=np.float64)
    
    valid = ~(np.isnan(obs) | np.isnan(sim))
    if not np.any(valid):
        return np.nan
    obs, sim = obs[valid], sim[valid]

    if metric_name == 'RMSE':
        return np.sqrt(np.nanmean((sim - obs) ** 2))
    elif metric_name == 'BIAS':
        return np.nanmean(sim - obs)
    elif metric_name == 'CORR':
        if np.std(obs) == 0 or np.std(sim) == 0:
            return np.nan
        return np.corrcoef(obs, sim)[0, 1]
    elif metric_name == 'KGE':
        r = np.corrcoef(obs, sim)[0, 1] if (np.std(obs) != 0 and np.std(sim) != 0) else np.nan
        if np.isnan(r) or np.mean(obs) == 0 or np.std(obs) == 0:
            return np.nan
        alpha = np.std(sim) / np.std(obs)
        beta = np.mean(sim) / np.mean(obs)
        return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
    else:
        raise ValueError(f"Unknown metric: {metric_name}")



In [ ]:
RUN_PREPROCESSING = False # 是否重新预处理数据

In [ ]:
# 数据预处理，可以跳过直接读取缓存数据
if RUN_PREPROCESSING:
    # 数据预处理
    import xarray as xr
    from tqdm import tqdm
    ResultDict = {
        'Obs': {},
        CtrlName: {},
        ExpName: {},
                }
    # ---------------------------------------- 读取观测数据 ----------------------------------------
    with tqdm(total=END_YEAR-START_YEAR+1, desc=f"Processing Obs") as pbar:
        for Year in range(START_YEAR, END_YEAR+1):
            # print(Year)
            Read_StartDate, Read_EndDate = datetime(Year, StartReadMonth, StartDay), datetime(Year, EndReadMonth, EndDay)  
            ResultDict['Obs'][Year] = {}
            for Var in LSM_VarList:
                # 读取观测数据
                ObsFolder = obs_folder_mapping[Var]
                ObsVarName = obs_var_mapping[Var]
                ObsFileName = f"{ObsFolder}_{ObsVarName}_10km.{Year}.nc"
                ObsFilePath = os.path.join(Obs_DataPath, ObsFolder, 'dst', ObsFileName)
                ds = xr.open_dataset(ObsFilePath)
                ResultDict['Obs'][Year][Var] = ds[ObsVarName].sel(time=slice(Read_StartDate, Read_EndDate)).values * obs_var_ScaleFactor[Var]
            pbar.update(1)    
    pbar.close()
    # ---------------------------------------- 读取模拟数据 ----------------------------------------
    ExpStates = [CtrlName, ExpName]
    from tqdm import tqdm
    for ExpState in ExpStates:
        with tqdm(total=END_YEAR-START_YEAR+1, desc=f"Processing {ExpState}") as pbar:
            for Year in range(START_YEAR, END_YEAR+1): 
                ResultDict[ExpState][Year] = {}
                for Var in LSM_VarList: 
                    Read_StartDate, Read_EndDate = datetime(Year, StartReadMonth, StartDay), datetime(Year, EndReadMonth, EndDay)  
                    SimVarName = sim_var_mapping[Var]
                    SimFilePath = os.path.join(Sim_DataPath, f"{ExpState}_{SimVarName}_10km.{Year}.nc")
                    ds = xr.open_dataset(SimFilePath)
                    ResultDict[ExpState][Year][Var] = ds[SimVarName].sel(time=slice(Read_StartDate, Read_EndDate)).values * sim_var_ScaleFactor[Var]
                pbar.update(1)           
    pbar.close()   
    # ------------------------------- 处理为气候态数据（时间平均，计算空间评分）-----------------------------
    ndays, ny, nx = ResultDict[ExpState][Year][Var].shape
    ClimateSpatialMeanResults = {}
    ClimateTemporalMeanResults = {}
    ClimateYearlyMeanResults = {} # 年际变率
    ClimateResults= {}
    # ResultDict[ExpName][2024]['Prec'].shape
    for ExpState in ['Obs', CtrlName, ExpName]:
        ClimateSpatialMeanResults[ExpState] = {}
        ClimateTemporalMeanResults[ExpState] = {}
        ClimateResults[ExpState] = {}
        for Var in LSM_VarList:
            TemplateClimateVar = np.zeros((YearNum, ndays, ny, nx))
            for YearIndex, Year in enumerate(range(START_YEAR, END_YEAR+1)):
                # print(f"{ExpState} {Var} {Year} shape:", ResultDict[ExpState][Year][Var].shape)
                TemplateClimateVar[YearIndex,:,:,:] = ResultDict[ExpState][Year][Var] 
            ClimateResults[ExpState][Var] = TemplateClimateVar # (YearNum, ndays, ny, nx)
            ClimateYearlyMeanResults[ExpState][Var] = np.nanmean(TemplateClimateVar, axis=1) # (YearNum, ny, nx)
            ClimateSpatialMeanResults[ExpState][Var] = np.nanmean(np.nanmean(TemplateClimateVar, axis=0), axis=0) # (ny, nx)
            ClimateTemporalMeanResults[ExpState][Var] = np.nanmean(TemplateClimateVar, axis=0) # (ndays, ny, nx)
    # --------------------------------- 缓存ClimateTemporalMeanResults数据 ----------------------------------------
    from netCDF4 import date2num
    Read_StartDate, Read_EndDate = datetime(2000, StartReadMonth, StartDay), datetime(2000, EndReadMonth, EndDay)
    DateList = pd.date_range(start=Read_StartDate, end=Read_EndDate, freq='24h')
    with tqdm(total=len(LSM_VarList), desc=f"Save cache data") as pbar:
        for Var in LSM_VarList:
            CachePath = os.path.join(DATA_PATH, 'Evaluation/cache/Grid', f'Annual_{Var}_10km.nc')
            with Dataset(CachePath, 'w') as cache_nc:
                cache_nc.createDimension('year', YearNum)
                cache_nc.createDimension('time', len(DateList))
                cache_nc.createDimension('lat', (lat.shape[0]))
                cache_nc.createDimension('lon', (lon.shape[0]))
                year_var = cache_nc.createVariable('year', 'i4', ('year',))
                time_var = cache_nc.createVariable('time', 'f4', ('time',))
                lat_var = cache_nc.createVariable('lat', 'f4', ('lat'))
                lon_var = cache_nc.createVariable('lon', 'f4', ('lon'))
                time_var.units = 'days since 1970-01-01 00:00:00'
                time_var.calendar = 'standard'
                # 写入数据
                for ExpState in ['Obs', CtrlName, ExpName]:
                    data_var = cache_nc.createVariable(f'{ExpState}_{Var}', 'f4', ('year', 'time', 'lat', 'lon'), zlib=True)
                    data_var[:] = ClimateResults[ExpState][Var]
                    data_var.units = VarUnit[Var]
                year_var[:] = np.arange(START_YEAR, END_YEAR+1)
                time_var[:] = date2num(DateList.to_pydatetime(), units=time_var.units, calendar=time_var.calendar)
                lat_var[:] = lat
                lon_var[:] = lon
            pbar.update(1)
else:
    ClimateSpatialMeanResults = {'Obs': {}, CtrlName: {}, ExpName: {}, f'{CtrlName}-Obs': {}, f'{ExpName}-Obs': {}, f'{ExpName}-{CtrlName}': {}}
    ClimateTemporalMeanResults = {'Obs': {}, CtrlName: {}, ExpName: {}, f'{CtrlName}-Obs': {}, f'{ExpName}-Obs': {}, f'{ExpName}-{CtrlName}': {}}
    ClimateYearlyMeanResults = {'Obs': {}, CtrlName: {}, ExpName: {}, f'{CtrlName}-Obs': {}, f'{ExpName}-Obs': {}, f'{ExpName}-{CtrlName}': {}}
    ClimateResults = {'Obs': {}, CtrlName: {}, ExpName: {}}
    # 直接读取缓存数据
    print("直接读取缓存数据...")
    for Var in LSM_VarList:
        CachePath = os.path.join(DATA_PATH, 'Evaluation/cache/Grid', f'Annual_{Var}_10km.nc')
        with Dataset(CachePath, 'r') as cache_nc:
            for ExpState in ['Obs', CtrlName, ExpName]:
                ClimateResults[ExpState][Var] = {}
                ClimateTemporalMeanResults[ExpState][Var] = {}
                ClimateSpatialMeanResults[ExpState][Var] = {}
                ClimateYearlyMeanResults[ExpState][Var] = {}
                ClimateResults[ExpState][Var] = cache_nc[f'{ExpState}_{Var}'][:]
                ClimateTemporalMeanResults[ExpState][Var] = np.nanmean(ClimateResults[ExpState][Var], axis=0)
                ClimateSpatialMeanResults[ExpState][Var] = np.nanmean(ClimateTemporalMeanResults[ExpState][Var], axis=0)
                ClimateYearlyMeanResults[ExpState][Var] = np.nanmean(ClimateResults[ExpState][Var], axis=1) # (YearNum, ny, nx)
    # 计算模拟和观测之间的差异
    for Var in LSM_VarList:
        ClimateTemporalMeanResults[f'{CtrlName}-Obs'][Var] = ClimateTemporalMeanResults[CtrlName][Var]-ClimateTemporalMeanResults['Obs'][Var]
        ClimateTemporalMeanResults[f'{ExpName}-Obs'][Var] = ClimateTemporalMeanResults[ExpName][Var]-ClimateTemporalMeanResults['Obs'][Var]
        ClimateTemporalMeanResults[f'{ExpName}-{CtrlName}'][Var] = ClimateTemporalMeanResults[ExpName][Var]-ClimateTemporalMeanResults[CtrlName][Var]  

        ClimateSpatialMeanResults[f'{CtrlName}-Obs'][Var] = ClimateSpatialMeanResults[CtrlName][Var]- ClimateSpatialMeanResults['Obs'][Var]  
        ClimateSpatialMeanResults[f'{ExpName}-Obs'][Var] = ClimateSpatialMeanResults[ExpName][Var]- ClimateSpatialMeanResults['Obs'][Var]
        ClimateSpatialMeanResults[f'{ExpName}-{CtrlName}'][Var] = ClimateSpatialMeanResults[ExpName][Var]- ClimateSpatialMeanResults[CtrlName][Var] 

        ClimateYearlyMeanResults[f'{CtrlName}-Obs'][Var] = ClimateYearlyMeanResults[CtrlName][Var]- ClimateYearlyMeanResults['Obs'][Var]
        ClimateYearlyMeanResults[f'{ExpName}-Obs'][Var] = ClimateYearlyMeanResults[ExpName][Var]- ClimateYearlyMeanResults['Obs'][Var]
        ClimateYearlyMeanResults[f'{ExpName}-{CtrlName}'][Var] = ClimateYearlyMeanResults[ExpName][Var]- ClimateYearlyMeanResults[CtrlName][Var]

In [ ]:
# ---------------------------------- 子区域时间序列分析 ---------------------------------
def extract_SubsetRegion_mask(var ,domain_mask):
    row_indices, col_indices = np.where(domain_mask == 1)  # 提取 ==1 的区域
    row_start, row_end = row_indices.min(), row_indices.max()
    col_start, col_end = col_indices.min(), col_indices.max()
    # 判断数组维数
    if len(var.shape) == 2:
        # 步骤2：切片获取子区域数据（保持二维结构）
        sub_var = var[row_start:row_end+1, col_start:col_end+1]
    elif len(var.shape) == 3:
        # 步骤2：切片获取子区域数据（保持三维结构）
        sub_var = var[:, row_start:row_end+1, col_start:col_end+1]
    elif len(var.shape) == 4:
        # 步骤2：切片获取子区域数据（保持四维结构）
        sub_var = var[:, :, row_start:row_end+1, col_start:col_end+1]
    else:
        raise ValueError("输入数据的维度不正确，请输入二维或三维数组。")
    return sub_var
import warnings
ExperimentTypes = ['Obs', CtrlName, ExpName]
SubRegionTemporalResults = {}
SubRegionSpatialResults = {}
SubRegionCilmateResults_3D = {}

for SubsetRegion in SubsetRegions:
    SubRegionTemporalResults[SubsetRegion] = {}
    SubRegionSpatialResults[SubsetRegion] = {}
    SubRegionCilmateResults_3D[SubsetRegion] = {}
    SubRegionDomainMask = create_domain_mask(lat_grid, lon_grid, SubsetRegions[SubsetRegion])
    for ExperimentType in ExperimentTypes:
        SubRegionTemporalResults[SubsetRegion][ExperimentType] = {}
        SubRegionSpatialResults[SubsetRegion][ExperimentType] = {}
        SubRegionCilmateResults_3D[SubsetRegion][ExperimentType] = {}

        for var in LSM_VarList:
            SubRegionVar = extract_SubsetRegion_mask(ClimateTemporalMeanResults[ExperimentType][var], SubRegionDomainMask) # 提取子区域变量
            SubRegionCilmateResults_3D[SubsetRegion][ExperimentType][var] = SubRegionVar # 计算子区域空间平均，得到 (time, lat, lon)
            SubRegionTemporalResults[SubsetRegion][ExperimentType][var] = np.nanmean(SubRegionVar, axis=(1,2)) # 计算子区域时间序列，得到 (time,)
            # --- 空间平均场：对时间平均 (axis=0)，可能触发警告 ---
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                spatial_mean = np.nanmean(SubRegionVar, axis=0)
            SubRegionSpatialResults[SubsetRegion][ExperimentType][var] = spatial_mean # 计算子区域空间平均，得到 (lat, lon)
# 计算年际标准差
AnnualStdResultsResults_3D = {}
for ExperimentType in ExperimentTypes:
    AnnualStdResultsResults_3D[ExperimentType] = {}
    for Var in LSM_VarList:
        AnnualStdResultsResults_3D[ExperimentType][Var] = np.nanstd(ClimateResults[ExpState][Var], axis=0)
# 计算子区域时间序列的年际标准差
SubRegionAnnualStd_TemporalResults = {}
for SubsetRegion in SubsetRegions:
    SubRegionAnnualStd_TemporalResults[SubsetRegion] = {}
    SubRegionDomainMask = create_domain_mask(lat_grid, lon_grid, SubsetRegions[SubsetRegion])
    for ExperimentType in ExperimentTypes:
        SubRegionAnnualStd_TemporalResults[SubsetRegion][ExperimentType] = {}
        for Var in LSM_VarList:
            SubRegionVar = extract_SubsetRegion_mask(AnnualStdResultsResults_3D[ExperimentType][Var], SubRegionDomainMask) # 提取子区域变量
            SubRegionAnnualStd_TemporalResults[SubsetRegion][ExperimentType][Var] = np.nanmean(SubRegionVar, axis=(1,2))


In [ ]:
# ---------------------------------- 子区域年际变率序列分析 ---------------------------------

import warnings
ExperimentTypes = ['Obs', CtrlName, ExpName]
Yearly_SubRegionTemporalResults = {
    'Mean': {},
    'Anomaly': {},
}

for SubsetRegion in SubsetRegions:
    Yearly_SubRegionTemporalResults['Mean'][SubsetRegion] = {}
    SubRegionDomainMask = create_domain_mask(lat_grid, lon_grid, SubsetRegions[SubsetRegion])
    for ExperimentType in ExperimentTypes:
        Yearly_SubRegionTemporalResults['Mean'][SubsetRegion][ExperimentType] = {}

        for var in LSM_VarList:
            SubRegionVar = extract_SubsetRegion_mask(ClimateYearlyMeanResults[ExperimentType][var], SubRegionDomainMask) # 提取子区域变量
            Yearly_SubRegionTemporalResults['Mean'][SubsetRegion][ExperimentType][var] = SubRegionVar # 计算子区域空间平均，得到 (year, lat, lon)
            Yearly_SubRegionTemporalResults['Mean'][SubsetRegion][ExperimentType][var] = np.nanmean(SubRegionVar, axis=(1,2)) # 计算子区域时间序列，得到 (year,)

# 计算距平
for SubsetRegion in SubsetRegions:
    Yearly_SubRegionTemporalResults['Anomaly'][SubsetRegion] = {}
    for ExperimentType in ExperimentTypes:
        Yearly_SubRegionTemporalResults['Anomaly'][SubsetRegion][ExperimentType] = {}
        for var in LSM_VarList:
            Yearly_SubRegionTemporalResults['Anomaly'][SubsetRegion][ExperimentType][var] = Yearly_SubRegionTemporalResults['Mean'][SubsetRegion][ExperimentType][var] - np.nanmean(Yearly_SubRegionTemporalResults['Mean'][SubsetRegion][ExperimentType][var], axis=0)


In [ ]:
# # # ================================== 子区域时间序列分析 ===================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
VarRange = {
    'SSM': [0, 0.55],
    'ET': [1, 5],
    'Q2m': [5, 21],
    'T2m': [290, 303],
    'PSFC': [960, 990],
    'Wind10m': [0, 6],
    'Prec': [0, 20],
}
DiffVarRange = {
    'SSM': [-0.2, 0.2],
    'ET': [-3, 3],
    'Q2m': [-5, 5],    
    'T2m': [-1, 5],
    'PSFC': [-0.5, 5],
    'Wind10m': [-0.5, 5],
    'Prec': [-10, 10],
}
plt.rcParams['axes.edgecolor'] = 'black'  # 边框颜色
plt.rcParams['axes.linewidth'] = 0.35     # 边框线宽
# --------------------------------- 绘图模块 ------------------------------------
def PlotVar(Plot_result_dict , Plot_std_result_dict ,VarName, subTitle,  ax1):
    # --------- 绘制土壤湿度模拟差异柱状图（左侧坐标轴）----------------------
    Year = 2000
    Read_StartDate, Read_EndDate = datetime(Year, StartReadMonth, StartDay), datetime(Year, EndReadMonth, EndDay)
    DateList = pd.date_range(start=Read_StartDate, end=Read_EndDate, freq='24h')
    ax1.set_ylim(VarRange[VarName][0], VarRange[VarName][1])
    ax1.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax1.set_ylabel(f"{VarName} ({VarUnit[VarName]})", fontsize=fontsize, labelpad =10)
    LineWidth = 2
    fill_between_alpha = 0.2
    # ----------- 绘制观测（右侧坐标轴）----------------------
    ax1.plot(DateList, Plot_result_dict['Obs'][VarName], 
            color= "#363636", marker='o',markersize= 3, markeredgewidth=0.1,
            linewidth = LineWidth , alpha=0.9, label="Obs", zorder=10) #,zorder=10  
    ax1.fill_between(DateList,
                     Plot_result_dict['Obs'][VarName] - Plot_std_result_dict['Obs'][VarName] * Std_Thresold ,
                Plot_result_dict['Obs'][VarName] + Plot_std_result_dict['Obs'][VarName] * Std_Thresold,
                color="gray", alpha = fill_between_alpha, zorder=10)
    ax1.plot(DateList, Plot_result_dict[CtrlName][VarName], 
            color= "#73A1C2", 
            linewidth = LineWidth , alpha=1, label='CTRL', zorder=10) #,zorder=10  
    ax1.fill_between(DateList, 
                     Plot_result_dict[CtrlName][VarName] - Plot_std_result_dict[CtrlName][VarName] * Std_Thresold ,
                Plot_result_dict[CtrlName][VarName] + Plot_std_result_dict[CtrlName][VarName] * Std_Thresold,
                color="#73A1C2", alpha = fill_between_alpha, zorder=11)
    # 年际变率阴影填充
    ax1.fill_between(DateList, 
                     Plot_result_dict[ExpName][VarName] - Plot_std_result_dict[ExpName][VarName] * Std_Thresold, 
                Plot_result_dict[ExpName][VarName] + Plot_std_result_dict[ExpName][VarName] * Std_Thresold,
                color="red", alpha = fill_between_alpha-0.1, zorder=12)
    ax1.plot(DateList, Plot_result_dict[ExpName][VarName], 
            color= "#FF8C69", 
            linewidth = LineWidth , alpha=1, label=ExpName, zorder=10) #,zorder=10  
    # 时间标签设置
    ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    # 设置X轴范围
    ax1.set_xlim(DateList[0], DateList[-2])
    # 设置刻度标签字体大小
    ax1.tick_params(axis='x', labelsize=fontsize-1)  # 设置x轴刻度标签大小
    ax1.tick_params(axis='y', labelsize=fontsize-1.5)  # 设置y轴刻度标签大小
    # 调整刻度线参数：
    ax1.minorticks_on()
    # 显示副刻度线
    ax1.tick_params(axis="both", which="major", direction="in", width=0.9, length=5, color='#363636')
    # grid 设置网格线性
    ax1.grid(True, which="major", linestyle="--", color="gray", linewidth=0.75, alpha = 0.3)

    ax1.text(0.05, 0.8, subTitle, fontsize= fontsize,
            va='bottom', ha='left', transform=ax1.transAxes, weight='normal',  alpha=1)
    pass

fontsize = 15
Std_Thresold = 0.2
TemporalAnalysisLSM_VarList = ['ET', 'Q2m', 'T2m', 'Prec']

Cols = len(SubsetRegions)
Rows = len(TemporalAnalysisLSM_VarList)
fig, axes = plt.subplots(Rows, Cols, figsize=(Cols*2.8*0.8, Rows*2.1*0.8), dpi=400, 
                        constrained_layout=True, sharex=True)
for IDy, VarName in enumerate(TemporalAnalysisLSM_VarList):
    for IDx, SubsetRegion in enumerate(SubsetRegions):
        axeIndex = IDy*Cols + IDx
        subTitle = f"({get_letter(axeIndex)})"
        PlotVar(SubRegionTemporalResults[SubsetRegion], SubRegionAnnualStd_TemporalResults[SubsetRegion], VarName, subTitle, axes[IDy, IDx])
        if IDx != 0:
            axes[IDy, IDx].set_yticklabels([])  # 清除 y 轴刻度标签
            axes[IDy, IDx].set_ylabel('')       # 清除 y 轴标签
        if IDy == 0:
            axes[IDy, IDx].set_title(SubsetRegion, fontsize= fontsize, x= 0.5, y = 1.03, loc='center', weight='normal',  alpha=1)
# 创建统一的图例
lines, labels = fig.axes[-1].get_legend_handles_labels()
fig.legend( lines, labels,   
           fontsize= fontsize-0.5,        
            bbox_to_anchor=(0.69, 0.01),ncol=4, framealpha=1, edgecolor='none',
)
# 设置子图间距
plt.subplots_adjust(wspace=0.1, hspace=0.2)  # 调整宽度和高度间距
plt.show()

In [ ]:
# # # ================================== 子区域年际变率时间序列分析 ===================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
MeanVarRange = {
    # 'SSM': [0, 0.55],
    'ET': [1, 5],
    'Q2m': [5, 25],
    'T2m': [292, 305],
    # 'PSFC': [960, 990],
    # 'Wind10m': [0, 6],
    'Prec': [0, 15],
}
DiffVarRange = {
    # 'SSM': [-0.2, 0.2],
    'ET': [-0.6, 0.6],
    'Q2m': [-2, 2],    
    'T2m': [-2, 2],
    # 'PSFC': [-0.5, 5],
    # 'Wind10m': [-0.5, 5],
    'Prec': [-4, 4],
}
MetricName = 'CORR'
Type = 'Mean'  # 'Mean' or 'Anomaly'
Type = 'Anomaly'  # 'Mean' or 'Anomaly'
if Type == 'Mean':
    VarRange = MeanVarRange.copy()
elif Type == 'Anomaly':
    VarRange = DiffVarRange.copy()
# --------------------------------- 绘图模块 ------------------------------------
def PlotYearVar(Plot_result_dict , VarRange, VarName, subTitle,  ax1):
    # --------- 绘制土壤湿度模拟差异柱状图（左侧坐标轴）----------------------

    Read_StartDate, Read_EndDate = datetime(2000, 1, 1), datetime(2024, 12, 31)
    DateList = pd.date_range(start=Read_StartDate, end=Read_EndDate, freq='YE')
    ax1.set_ylim(VarRange[VarName][0], VarRange[VarName][1])
    # print(VarName, VarRange[VarName][0], VarRange[VarName][1], 
        #   np.min(Plot_result_dict[CtrlName][VarName]), np.max(Plot_result_dict[CtrlName][VarName]))
    ax1.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax1.set_ylabel(f"{VarName} ({VarUnit[VarName]})", fontsize=fontsize, labelpad =10)
    LineWidth = 3
    fill_between_alpha = 0.2
    # ----------- 绘制观测（右侧坐标轴）----------------------
    ax1.plot(DateList, Plot_result_dict['Obs'][VarName], 
            color= "#363636", marker='o',markersize= 3, markeredgewidth=0.1,
            linewidth = LineWidth , alpha=0.8, label="Obs", zorder=10) #,zorder=10  

    ax1.plot(DateList, Plot_result_dict[CtrlName][VarName], 
            color= "#73A1C2", 
            linewidth = LineWidth , alpha=1, label='CTRL', zorder=10, linestyle='--') #,zorder=10  

    ax1.plot(DateList, Plot_result_dict[ExpName][VarName], 
            color= "#FF8C69", 
            linewidth = LineWidth , alpha=1, label=ExpName, zorder=10) #,zorder=10  
    #如果是距平序列，则标出0线
    if Type == 'Anomaly':
        ax1.axhline(y=0, color='#FF0000', linestyle='--', linewidth=1.2, alpha=0.7)

    # 计算评分
    
    obs = Plot_result_dict['Obs'][VarName]
    CtrlSim = Plot_result_dict[CtrlName][VarName]
    ExpSim = Plot_result_dict[ExpName][VarName]
    CtrlMetric, ExpMetric = compute_metric(obs, CtrlSim, MetricName), compute_metric(obs, ExpSim, MetricName)
    ax1.text(0.5, 0.31, f"CTRL: {CtrlMetric:.2f}", # f"{MetricName}={CtrlMetric:.2f}"
             fontsize=fontsize-2., color="#009ACD", 
             bbox=dict(facecolor='none', edgecolor='none', boxstyle='round,pad=0.5', alpha=1), 
             transform=ax1.transAxes, va='top')
    ax1.text(0.5, 0.16, f"FSM: {ExpMetric:.2f}", # f"{MetricName}={CtrlMetric:.2f}"
             fontsize=fontsize-2., color="#FF4500", 
             bbox=dict(facecolor='none', edgecolor='none', boxstyle='round,pad=0.5', alpha=1), 
             transform=ax1.transAxes, va='top')
    # 时间标签设置
    ax1.minorticks_on()
    ax1.xaxis.set_major_locator(mdates.YearLocator(base=10))
    ax1.xaxis.set_minor_locator(mdates.YearLocator(5))
    # ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    # 旋转45度
    ax1.tick_params(axis='x', labelsize=fontsize-1, rotation=45)
    # 设置X轴范围
    ax1.set_xlim(datetime(2000, 1, 1), datetime(2024, 12, 31))
    # 设置刻度标签字体大小
    ax1.tick_params(axis='x', labelsize=fontsize-1.5)  # 设置x轴刻度标签大小
    ax1.tick_params(axis='y', labelsize=fontsize-1.5)  # 设置y轴刻度标签大小
    # 调整刻度线参数：
    ax1.minorticks_on()
    # 显示副刻度线
    ax1.tick_params(axis="both", which="major", direction="in", width=1, length=5, color='#363636')
    # grid 设置网格线性
    ax1.grid(True, which="major", linestyle="--", color="gray", linewidth=0.75, alpha = 0.3)
    
    # ax1.set_title(subTitle, fontsize= fontsize, x= 0.01, y = 1.03, 
    #               loc='left', weight='normal',  alpha=1)
    ax1.text(0.05, 0.8, subTitle, fontsize= fontsize,
            va='bottom', ha='left', transform=ax1.transAxes, weight='normal',  alpha=1)

    pass

fontsize = 15
Std_Thresold = 0.2
TemporalAnalysisLSM_VarList = ['ET', 'Q2m', 'T2m', 'Prec']
# 修改部分1：在开始绘图前设置全局rcParam参数
plt.rcParams['axes.edgecolor'] = 'black'  # 边框颜色
plt.rcParams['axes.linewidth'] = 0.35     # 边框线宽
Cols = len(SubsetRegions)
Rows = len(TemporalAnalysisLSM_VarList)
fig, axes = plt.subplots(Rows, Cols, figsize=(Cols*2.8*0.88, Rows*2.1*0.88), dpi=400, 
                        constrained_layout=True, sharex=True)
for IDy, VarName in enumerate(TemporalAnalysisLSM_VarList):
    for IDx, SubsetRegion in enumerate(SubsetRegions):
        axeIndex = IDy*Cols + IDx
        subTitle = f"({get_letter(axeIndex)})"
        PlotYearVar(Yearly_SubRegionTemporalResults[Type][SubsetRegion], VarRange, VarName, subTitle, axes[IDy, IDx])
        if IDx != 0:
            axes[IDy, IDx].set_yticklabels([])  # 清除 y 轴刻度标签
            axes[IDy, IDx].set_ylabel('')       # 清除 y 轴标签
        if IDy == 0:
            axes[IDy, IDx].set_title(SubsetRegion, fontsize= fontsize, x= 0.5, y = 1.03, loc='center', weight='normal',  alpha=1)

# 创建统一的图例
lines, labels = fig.axes[-1].get_legend_handles_labels()
fig.legend( lines, labels,   
           fontsize= fontsize-0.5,        
            bbox_to_anchor=(0.66, 0.01),ncol=4, framealpha=1, edgecolor='none',
)
# 设置子图间距
plt.subplots_adjust(wspace=0.08, hspace=0.2)  # 调整宽度和高度间距
plt.show()

In [ ]:
# ---------------------------------- 子区域空间pattern分析 ---------------------------------

SubRegionSpatialResults = {}
for RegionName in SubsetRegions:
    SubRegionSpatialResults[RegionName] = {}
    SubRegionDomainMask = create_domain_mask(lat_grid, lon_grid, SubsetRegions[RegionName])
    for ExperimentType in ExperimentTypes:
        SubRegionSpatialResults[RegionName][ExperimentType] = {}
        for var in LSM_VarList:
            SubRegionVar = extract_SubsetRegion_mask(ClimateSpatialMeanResults[ExperimentType][var], SubRegionDomainMask) # 提取子区域变量
            SubRegionSpatialResults[RegionName][ExperimentType][var] = SubRegionVar


In [ ]:
# # # ================================== 绘制泰勒图 ===================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
from scipy import stats
from matplotlib.projections import PolarAxes
from mpl_toolkits.axisartist import floating_axes
from mpl_toolkits.axisartist import grid_finder

# 设置JGR-A期刊的绘图风格
# plt.rcParams['font.family'] = 'serif'
# plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['xtick.major.width'] = 1.2
plt.rcParams['ytick.major.width'] = 1.2
MaxRange=2
def taylor_diagram(obs_data, sim_data_ctrl, sim_data_set, var_names, ax=None, fontsize=12):
    """
    绘制泰勒图
    
    Parameters:
    -----------
    obs_data : dict
        观测数据字典，变量名为键，气候态数据为值（2D数组）
    sim_data_ctrl : dict
        控制实验模拟数据字典，变量名为键，气候态数据为值（2D数组）
    sim_data_set : dict
        设置实验模拟数据字典，变量名为键，气候态数据为值（2D数组）
    var_names : list
        变量名称列表
    ax : matplotlib axes object, optional
        泰勒图专用的轴对象，如果提供则直接使用；否则创建新的图形和轴
    fontsize : int, optional
        字体大小，默认为12
    """
    cmap = cm.turbo
    
    # 如果没有提供ax，则创建一个新的图形和轴
    if ax is None:
        fig = plt.figure(figsize=(5, 5), dpi=400)
        trans = PolarAxes.PolarTransform()
        r1_locs = np.hstack((np.arange(1,10)/10.0,[0.95, 0.99]))
        t1_locs = np.arccos(r1_locs)        
        gl1 = grid_finder.FixedLocator(t1_locs)    
        tf1 = grid_finder.DictFormatter(dict(zip(t1_locs, map(str, r1_locs))))
        # r2_locs = np.arange(0, 2, 0.5)
        # r2_labels = ['0 ', '0.25 ', '0.50 ', '0.75 ', 'REF ', '1.25 ', '1.50 ', '2' ] # 
        # gl2 = grid_finder.FixedLocator(r2_locs)
        # tf2 = grid_finder.DictFormatter(dict(zip(r2_locs, map(str, r2_labels))))
        ghelper = floating_axes.GridHelperCurveLinear(trans, extremes=(0, np.pi/2, 0, 2),
                                                      grid_locator1=gl1, tick_formatter1=tf1,
                                                      grid_locator2=gl2, tick_formatter2=tf2)
        ax = floating_axes.FloatingSubplot(fig, 111, grid_helper=ghelper)
        fig.add_subplot(ax)
        created_fig = True
    else:
        # 用户提供了自定义的泰勒图轴对象
        created_fig = False
        # 确保传入的是正确的泰勒图轴
        trans = PolarAxes.PolarTransform()
    
    # 计算统计量
    stats_dict = {}
    
    for var in var_names:
        # 处理控制实验数据: Ctrl
        if var in obs_data and var in sim_data_ctrl:
            obs = obs_data[var].flatten()
            sim_ctrl = sim_data_ctrl[var].flatten()
            
            # 移除NaN值
            mask = ~(np.isnan(obs) | np.isnan(sim_ctrl))
            obs_clean = obs[mask]
            sim_ctrl_clean = sim_ctrl[mask]
            
            if len(obs_clean) > 0:
                # 计算相关系数
                corr_ctrl = np.corrcoef(obs_clean, sim_ctrl_clean)[0, 1]
                
                # 计算标准差比率（模拟/观测）
                std_obs = np.std(obs_clean)
                std_sim_ctrl = np.std(sim_ctrl_clean)
                std_ratio_ctrl = std_sim_ctrl / std_obs if std_obs > 0 else np.nan
                
                # 计算中心化均方根误差
                bias_ctrl = np.mean(sim_ctrl_clean) - np.mean(obs_clean)
                centered_rmse_ctrl = np.sqrt(np.mean((sim_ctrl_clean - np.mean(sim_ctrl_clean) - 
                                                    (obs_clean - np.mean(obs_clean)))**2))
                normalized_rmse_ctrl = centered_rmse_ctrl / std_obs if std_obs > 0 else np.nan
                
                stats_dict[f'{var}_{CtrlName}'] = {
                    'correlation': corr_ctrl,
                    'std_ratio': std_ratio_ctrl,
                    'normalized_rmse': normalized_rmse_ctrl,
                    'bias': bias_ctrl,
                    'std_obs': std_obs,
                    'std_sim': std_sim_ctrl,
                    'experiment_type': 'control'
                }
        
        # 处理设置实验数据: WCSMX
        if var in obs_data and var in sim_data_set:
            obs = obs_data[var].flatten()
            sim_set = sim_data_set[var].flatten()
            
            # 移除NaN值
            mask = ~(np.isnan(obs) | np.isnan(sim_set))
            obs_clean = obs[mask]
            sim_set_clean = sim_set[mask]
            
            if len(obs_clean) > 0:
                # 计算相关系数
                corr_set = np.corrcoef(obs_clean, sim_set_clean)[0, 1]
                
                # 计算标准差比率（模拟/观测）
                std_obs = np.std(obs_clean)
                std_sim_set = np.std(sim_set_clean)
                std_ratio_set = std_sim_set / std_obs if std_obs > 0 else np.nan
                
                # 计算中心化均方根误差
                bias_set = np.mean(sim_set_clean) - np.mean(obs_clean)
                centered_rmse_set = np.sqrt(np.mean((sim_set_clean - np.mean(sim_set_clean) - 
                                                   (obs_clean - np.mean(obs_clean)))**2))
                normalized_rmse_set = centered_rmse_set / std_obs if std_obs > 0 else np.nan
                
                stats_dict[f'{var}_{ExpName}'] = {
                    'correlation': corr_set,
                    'std_ratio': std_ratio_set,
                    'normalized_rmse': normalized_rmse_set,
                    'bias': bias_set,
                    'std_obs': std_obs,
                    'std_sim': std_sim_set,
                    'experiment_type': 'setting'
                }

    # 添加观测数据点
    obs_std_mean = np.nanmean([np.std(obs_data[var].flatten()) for var in var_names if var in obs_data])
    stats_dict['Obs'] = {
        'correlation': 1.0,
        'std_ratio': 1.0,
        'normalized_rmse': 0.0,
        'bias': 0.0,
        'std_obs': obs_std_mean,
        'std_sim': obs_std_mean,
        'experiment_type': 'obs'
    }
    
    # 设置泰勒图的轴属性
    ax.axis["top"].set_axis_direction("bottom")  
    ax.axis["top"].toggle(ticklabels=True, label=True)
    ax.axis["top"].major_ticklabels.set_axis_direction("top")
    ax.axis["top"].label.set_axis_direction("top")
    ax.axis["top"].label.set_text("Correlation")
    ax.axis["top"].label.set_fontsize(fontsize+2)
    ax.axis["left"].set_axis_direction("bottom") 
    ax.axis["left"].label.set_text("Standard deviation")
    ax.axis["left"].label.set_fontsize(fontsize+2)
    ax.axis["right"].set_axis_direction("top")   
    ax.axis["right"].toggle(ticklabels=True)
    ax.axis["right"].major_ticklabels.set_axis_direction("left")
    ax.axis["bottom"].set_visible(False)   
    ax.axis["bottom"].major_ticklabels.set_rotation(45)      
    ax.grid(True)
    polar_ax = ax.get_aux_axes(trans)   

    rs, ts = np.meshgrid(np.linspace(0, MaxRange, 100),
                         np.linspace(0, np.pi/2, 100))
    rms = np.sqrt(1 + rs**2 - 2*rs*np.cos(ts))
    CS = polar_ax.contour(ts, rs, rms, colors='gray', linestyles='--')
    plt.clabel(CS, inline=1, fontsize=fontsize-1)
    t = np.linspace(0, np.pi/2)
    r = np.zeros_like(t) + 1
    polar_ax.plot(t, r, 'k--')
    polar_ax.text(np.pi/2+0.032, 1.02, " 1.00", size=fontsize-1, ha="right", va="top",
                  bbox=dict(boxstyle="square", ec='w', fc='w'))

    # 设置颜色映射
    # unique_vars = list(set([name.split('_')[0] for name in stats_dict.keys() if name != 'Obs']))
    seen = set()
    unique_vars = []
    for name in stats_dict.keys():
        if name == 'Obs':
            continue
        var = name.split('_')[0]
        if var not in seen:
            unique_vars.append(var)
            seen.add(var)
    colors = cmap(np.linspace(0, 1, len(unique_vars)))
    color_dict = {}
    
    # 为每个变量分配颜色
    for i, var in enumerate(unique_vars):
        if f'{var}_{CtrlName}' in stats_dict:
            color_dict[f'{var}_{CtrlName}'] = colors[i]
        if f'{var}_{ExpName}' in stats_dict:
            color_dict[f'{var}_{ExpName}'] = colors[i]
    
    # 为观测点单独设置颜色
    color_dict['Obs'] = 'red'

    # 分别绘制控制实验和设置实验的点
    for var_name, stats in stats_dict.items():
        if var_name == 'Obs':
            # 绘制观测点
            angle = np.arccos(stats['correlation'])
            radius = stats['std_ratio']
            polar_ax.scatter(angle, radius, c=color_dict[var_name], 
                      marker='^', s=150, label='Obs', 
                      edgecolors='black', alpha=0.8, linewidth=2)
        elif not np.isnan(stats['correlation']) and not np.isnan(stats['std_ratio']):
            # 计算极坐标角度（相关系数转换为角度）
            angle = np.arccos(stats['correlation'])
            radius = stats['std_ratio']
            
            # 根据实验类型选择标记
            if stats['experiment_type'] == 'control':
                marker = 's'  # 方形表示控制实验
                label = var_name.replace(f'_{CtrlName}', '') 
            elif stats['experiment_type'] == 'setting':
                marker = 'o'  # 圆形表示设置实验
                label = var_name.replace(f'_{ExpName}', '') 
            # else:
            #     marker = 'o'
            #     label = var_name
            
            # 绘制点
            polar_ax.scatter(angle, radius, c=[color_dict[var_name]], 
                      marker=marker, s=150, label=label, 
                      edgecolors='black', alpha=0.6, linewidth=2)

    if created_fig:
        plt.tight_layout()
        return fig, stats_dict
    else:
        return stats_dict
# ----------------------------------------------------------------
TaylorPlotSubsetRegions = {
    # 'ECN': [21.5, 40, 108, 121], # Eastern China
    'HHP':[35, 40, 114., 119],
    'LPT': [35, 40, 108, 114.],
    'QDW': [28.5, 35, 108, 112.5], # Qinling-Daba-Wushan
    'YHP': [28.5, 35, 112.5, 121], 
    'SEH': [23, 28.5, 110, 120], # Southeastern Hills

}
fontsize = 14
TaylorPlotVarList = ['Q2m', 'Prec', 'ET','Wind10m', 'T2m', 'PSFC' ]
for IDRegion, RegionName in enumerate(TaylorPlotSubsetRegions) :
    # 正确的使用方式示例：
    # 创建泰勒图专用的轴
    fig = plt.figure(figsize=(3.8, 3.8), dpi=400)
    trans = PolarAxes.PolarTransform()
    r1_locs = np.hstack((np.arange(1,10)/5,[0.95]))
    t1_locs = np.arccos(r1_locs)        
    gl1 = grid_finder.FixedLocator(t1_locs)    
    tf1 = grid_finder.DictFormatter(dict(zip(t1_locs, map(str, r1_locs))))
    r2_locs = np.arange(0, MaxRange, 0.5)
    r2_labels = ['0 ', '0.25 ', '0.50 ', '0.75 ', 'REF ', '1.25 ', '1.50 ', '1.75'] #['0 ', '0.25 ', '0.50 ', '0.75 ', 'REF ', '1.25 ', '1.50 ', '1.75']
    gl2 = grid_finder.FixedLocator(r2_locs)
    tf2 = grid_finder.DictFormatter(dict(zip(r2_locs, map(str, r2_labels))))
    ghelper = floating_axes.GridHelperCurveLinear(trans, extremes=(0, np.pi/2, 0, MaxRange),
                                                grid_locator1=gl1, tick_formatter1=tf1,
                                                grid_locator2=gl2, tick_formatter2=tf2)
    
    ax = floating_axes.FloatingSubplot(fig, 111, grid_helper=ghelper)
    
    fig.add_subplot(ax)

    # 使用函数绘制泰勒图
    stats_results = taylor_diagram(
        SubRegionTemporalResults[RegionName]['Obs'], 
        SubRegionTemporalResults[RegionName][CtrlName], 
        SubRegionTemporalResults[RegionName][ExpName], 
        TaylorPlotVarList,
        ax=ax,
        fontsize=fontsize
    )
    # 添加图例
    # 确保每个实验类型的标签是唯一的，避免重复显示
    handles, labels = ax.get_legend_handles_labels()
    # 创建一个字典，确保标签唯一
    by_label = dict(zip(labels, handles))
    ax.text(1, 1., f'({get_letter(IDRegion)}) {RegionName}', transform=ax.transAxes, 
            ha='center', va='center', fontsize=fontsize+2,
            fontweight='bold'
            )
    

    # 显示图形
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
# 创建一个新图形，并隐藏坐标轴
fig, ax = plt.subplots(figsize=(3, 2), dpi=400)  # 调整宽度高度以适配图例
ax.axis('off')  # 完全隐藏轴

# 使用 fig.legend 直接在图形上绘制图例（不依赖 ax）
legend = fig.legend(
    handles=by_label.values(),
    labels=by_label.keys(),
    loc='center',          # 放在图形中央
    ncol=2,                # 一行 7 列
    fontsize=9,
    markerscale=0.8,
    handletextpad=0.2,
    labelspacing=1,
    columnspacing = 0.6,
    frameon=False
)
plt.show()

In [ ]:
# ================================ 计算气候态评分 ==============================
METRICS = ['RMSE', 'BIAS', 'CORR', 'KGE']
SubRegionsList = list(SubsetRegions.keys())
EXPs = [CtrlName, ExpName]
# ============================ 计算气候态空间评分 =============================
for Metric in METRICS:
    S_Metrics_df = pd.DataFrame(index=LSM_VarList)
    for ExpState in EXPs:
        for RegionName in SubRegionsList:
            for Var in LSM_VarList:
                sim_data = (SubRegionSpatialResults[RegionName][ExpState][Var]).flatten()  
                obs_data = (SubRegionSpatialResults[RegionName]['Obs'][Var]).flatten()  
                # print(ExpState, RegionName, Var, np.nanmax(sim_data), np.nanmax(obs_data))
                # 质控
                sim_data[sim_data>10000]=np.nan
                obs_data[obs_data>10000]=np.nan
                if Var == 'SM':
                    sim_data[sim_data>0.6]=np.nan
                    obs_data[obs_data>0.6]=np.nan       
                # 计算指标
                MetricValue = compute_metric(obs_data, sim_data, Metric)
                S_Metrics_df.loc[Var, f'{RegionName} ({ExpState})'] = (MetricValue)
    SaveFlieName = f"LSMvar_ClimateMean_Grid_SpatialMetrics_{Metric}.xlsx"
    SavePath = '/home/yfdong/data/work/SMP/code/AnalysisModule/Evaluation/MetricExcel'
    S_Metrics_df.to_excel(os.path.join(SavePath, SaveFlieName))
    # print(S_Metrics_df)
# ============================ 计算气候态时间评分 =============================
for Metric in METRICS:
    T_Metrics_df = pd.DataFrame(index=LSM_VarList)
    for ExpState in EXPs:
        for RegionName in SubRegionsList:
            for Var in LSM_VarList:
                sim_data = (SubRegionTemporalResults[RegionName][ExpState][Var]).flatten()  
                obs_data = (SubRegionTemporalResults[RegionName]['Obs'][Var]).flatten()
                # 质控
                sim_data[sim_data>10000]=np.nan
                obs_data[obs_data>10000]=np.nan
                if Var == 'SM':
                    sim_data[sim_data>0.6]=np.nan
                    obs_data[obs_data>0.6]=np.nan
                # 计算指标
                MetricValue = compute_metric(obs_data, sim_data, Metric)
                T_Metrics_df.loc[Var, f'{RegionName} ({ExpState})'] = (MetricValue)
    SaveFlieName = f"LSMvar_ClimateMean_Grid_TemporalMetrics_{Metric}.xlsx"
    SavePath = '/home/yfdong/data/work/SMP/code/AnalysisModule/Evaluation/MetricExcel'
    T_Metrics_df.to_excel(os.path.join(SavePath, SaveFlieName))